In [29]:
using LowLevelFEM

In [30]:
openGeometry("periodic-2D.geo")
#openPreProcessor()

In [31]:
mat = Material("body")

U = Field([mat], type=:VectorField, dim=2, fieldName=:u);
Φ = Field([mat], type=:ScalarField, dim=2, fieldName=:φ);

In [32]:
Ku = ∫(ε(U)' ⋅ D(:PlaneStress, mat) ⋅ ε(U));

In [33]:
bc_u = BoundaryCondition("P", ux=0, uy=0, field=U)
bc_φ = BoundaryCondition("P", φ=0.01, field=Φ)


periodic1 = MPC(master="Q", slave="P", field=U)
periodic2 = MPC(master="Q", slave="P", field=Φ)
mpc_u1 = MPC(master="P", slave="left", field=U)
mpc_φ1 = MPC(master="P", slave="left", field=Φ);
mpc_u2 = MPC(master="Q", slave="right", field=U)
mpc_φ2 = MPC(master="Q", slave="right", field=Φ);

In [34]:
R = rigidRotationMap(mpc_u1, mpc_φ1);
R += rigidRotationMap(mpc_u2, mpc_φ2);

In [35]:
Kuφ = Ku * R
Kφ = R' * Ku * R

K = SystemMatrix([Ku Kuφ; Kuφ' Kφ]);

In [36]:
fu = ∫(U ⋅ [0,-10])
fφ = ∫(Φ ⋅ 0, Γ="P")

F = SystemVector([fu, fφ]);

In [37]:
u, φ = solveField(K, F, support=[bc_u, bc_φ], mpc=[mpc_u1, mpc_φ1, mpc_u2, mpc_φ2, periodic1, periodic2])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; -0.0003048181132314054; 0.0024358358179708012;;], [0.0], Int64[], 1, :v2D, Problem("periodic-2D", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 361, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)), ScalarField(Matrix{Float64}[], [0.01; 0.01; … ; 0.0; 0.0;;], [0.0], Int64[], 1, :scalar, Problem("periodic-2D", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 361, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false)))

In [38]:
showDoFResults(u + R * φ, name="u", visible=true, factor=20)

0

In [39]:
for name in ("P", "Q")
    tag = LowLevelFEM.getTagForPhysicalName(name)
    nodes, coord =
        gmsh.model.mesh.getNodesForPhysicalGroup(-1, tag)

    @show name nodes coord
end

name = "P"
nodes = UInt64[0x0000000000000005]
coord = [0.0, 0.5, 0.0]
name = "Q"
nodes = UInt64[0x0000000000000006]
coord = [10.0, 0.5, 0.0]


In [40]:
tagQ = LowLevelFEM.getTagForPhysicalName("Q")
_, qcoord =
    gmsh.model.mesh.getNodesForPhysicalGroup(-1, tagQ)

tagR = LowLevelFEM.getTagForPhysicalName("right")
_, rcoord =
    gmsh.model.mesh.getNodesForPhysicalGroup(-1, tagR)

xQ = qcoord[1]

@show xQ
@show extrema(rcoord[1:3:end] .- xQ)

xQ = 10.0
extrema(rcoord[1:3:end] .- xQ) = (0.0, 0.0)


(0.0, 0.0)

In [41]:
openPostProcessor()